In [1]:
import h5py
import numpy as np
import os

train_filename = 'train_signs.h5'
test_filename = 'test_signs.h5'

if os.path.exists(train_filename) and os.path.exists(test_filename):
    print("Файли знайдено")
else:
    print("Файли не знайдено")

Файли знайдено


In [3]:
def load_dataset():
    train_dataset = h5py.File(train_filename, "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:]) 
    train_set_y_orig = np.array(train_dataset["train_set_y"][:]) 

    test_dataset = h5py.File(test_filename, "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:]) 
    test_set_y_orig = np.array(test_dataset["test_set_y"][:]) 

    classes = np.array(test_dataset["list_classes"][:]) 
    
    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))
    
    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes

print("Функція load_dataset() створена")

Функція load_dataset() створена


In [4]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_dataset()

X_train = X_train_orig / 255.
X_test = X_test_orig / 255.


Y_train = Y_train_orig.T
Y_test = Y_test_orig.T

print("\РЕЗУЛЬТАТ")
print(f"Кількість тренувальних картинок: {X_train.shape[0]}")
print(f"Кількість тестових картинок: {X_test.shape[0]}")
print(f"Розмір однієї картинки: {X_train.shape[1]}x{X_train.shape[2]} пікселів, {X_train.shape[3]} кольорові канали.")

\РЕЗУЛЬТАТ
Кількість тренувальних картинок: 1080
Кількість тестових картинок: 120
Розмір однієї картинки: 64x64 пікселів, 3 кольорові канали.


In [5]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model

def HandSignModel(input_shape):
    X_input = Input(input_shape)
    X = Conv2D(32, (3, 3), activation='relu', name='conv1')(X_input)
    X = MaxPooling2D((2, 2), name='maxpool1')(X)

    X = Conv2D(64, (3, 3), activation='relu', name='conv2')(X)
    X = MaxPooling2D((2, 2), name='maxpool2')(X)

    X = Conv2D(128, (3, 3), activation='relu', name='conv3')(X)
    X = MaxPooling2D((2, 2), name='maxpool3')(X)
    
    X = Flatten()(X)
    X = Dense(6, activation='softmax', name='output_layer')(X)
    model = Model(inputs=X_input, outputs=X, name='HandSignModel')
    return model

sign_model = HandSignModel((64, 64, 3))
sign_model.summary()

C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


Model: "HandSignModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 64, 64, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1 (Conv2D)                       │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ maxpool1 (MaxPooling2D)              │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2 (Conv2D)                       │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ maxpool2 (MaxPooling2D)              │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3 (Conv2D)                       │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ maxpool3 (MaxPooling2D)              │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output_layer (Dense)                 │ (None, 6)                   │          27,654 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 120,902 (472.27 KB)

 Trainable params: 120,902 (472.27 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
sign_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy'] 
)
print("Модель скомпільована")

print("ТРЕНУВАННЯ")

history = sign_model.fit(
    x=X_train, 
    y=Y_train, 
    epochs=25, 
    batch_size=32, 
    validation_data=(X_test, Y_test) 
)

print("завершено")

Модель скомпільована
ТРЕНУВАННЯ
Epoch 1/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.1917 - loss: 1.7810 - val_accuracy: 0.2667 - val_loss: 1.7410
Epoch 2/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.4694 - loss: 1.3962 - val_accuracy: 0.6750 - val_loss: 1.1263
Epoch 3/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.6685 - loss: 0.9348 - val_accuracy: 0.7667 - val_loss: 0.7973
Epoch 4/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.7343 - loss: 0.7797 - val_accuracy: 0.7917 - val_loss: 0.6389
Epoch 5/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.8259 - loss: 0.5398 - val_accuracy: 0.7917 - val_loss: 0.5622
Epoch 6/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.8324 - loss: 0.4799 - val_accuracy: 0.8250 - val_loss: 0.5319
Epoch 7/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.8759 - loss: 0.3606 - val_accuracy: 0.8917 - val_loss: 0.3262
Epoch 8/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9074 - loss: 0

In [7]:
preds = sign_model.evaluate(X_test, Y_test)

print("РЕЗУЛЬТАТИ")
print(f"Помилка (Loss): {preds[0]:.4f}")
print(f"Точність (Test Accuracy): {preds[1]:.4f} (або {preds[1]*100:.2f}%)")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9417 - loss: 0.1746
РЕЗУЛЬТАТИ
Помилка (Loss): 0.1746
Точність (Test Accuracy): 0.9417 (або 94.17%)
